# Resort Churn — ExtraTreesClassifier

Minimal-preprocessing baseline using `ExtraTreesClassifier`.

- **Train:** `resort_train.csv` (from the 869_course repo)
- **Test:** `resort_test.csv`
- **Metric:** macro F1 (via stratified cross-validation)
- **Output:** `submission.csv` with `GuestID` and 0/1 `Churned` predictions

Feature engineering is intentionally kept to the bare minimum needed for the model to run:
drop the ID / free-text columns, median-impute numerics, and one-hot encode the
low-cardinality categoricals.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.base import clone

RANDOM_STATE = 42

In [2]:
# Load data straight from the course repo (raw URLs)
TRAIN_URL = "https://raw.githubusercontent.com/stepthom/869_course/main/data/resort_train.csv"
TEST_URL = "https://raw.githubusercontent.com/stepthom/869_course/main/data/resort_test.csv"

train = pd.read_csv(TRAIN_URL)
test = pd.read_csv(TEST_URL)

print("train:", train.shape, "| test:", test.shape)
train.head()

train: (6954, 21) | test: (1739, 20)


,GuestID,BookingDate,PromoCode,Region,AllInclusive,Room,PackageType,Age,VIP,RoomService,...,Retail,Spa,Entertainment,LoyaltyPoints,SurveyScore,DaysSinceEmail,BookingChannel,AgeGroup,ReferralSource,Churned
0,619623,2024-02-10,NaN,Americas,0.0,G/630/S,Relaxation,0.0,0.0,0.0,...,0.0,0.0,0.0,6915,5,136,Website,NaN,Facebook,1
1,776250,2024-01-03,NaN,Americas,1.0,G/201/S,Relaxation,17.0,0.0,0.0,...,0.0,0.0,0.0,8571,5,362,Corporate,Minor,Billboard,1
2,932709,2023-01-17,NaN,Americas,NaN,G/1483/S,Wellness,35.0,0.0,0.0,...,0.0,0.0,0.0,1142,4,154,TravelAgent,Middle,Facebook,0
3,771839,2023-12-09,PromoA,Europe,1.0,D/164/S,Adventure,26.0,0.0,0.0,...,0.0,NaN,0.0,9642,2,128,Website,Young,Magazine,1
4,755501,2024-02-15,PromoA,Americas,0.0,G/818/P,Relaxation,13.0,0.0,0.0,...,60.0,1.0,5147.0,5528,4,35,Mobile,Minor,Google,0


In [3]:
TARGET = "Churned"
ID_COL = "GuestID"

# Drop the identifier, the raw date, and the very high-cardinality room code.
# Everything else is fed to the model as-is.
DROP_COLS = [ID_COL, "BookingDate", "Room"]

X = train.drop(columns=DROP_COLS + [TARGET])
y = train[TARGET]
X_test = test.drop(columns=DROP_COLS)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("numeric:", numeric_cols)
print("categorical:", categorical_cols)

numeric: ['AllInclusive', 'Age', 'VIP', 'RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment', 'LoyaltyPoints', 'SurveyScore', 'DaysSinceEmail']
categorical: ['PromoCode', 'Region', 'PackageType', 'BookingChannel', 'AgeGroup', 'ReferralSource']


In [ ]:
# Minimal preprocessing: impute then one-hot encode categoricals.
# ExtraTrees can't take NaNs or strings, so this is the smallest transform that works.
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("clf", ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])

In [5]:
# 5-fold stratified cross-validation, scored on macro F1.
# We loop manually so we can keep the validation predictions from the best-scoring
# fold and build a classification report for that candidate below.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_scores = []
best = {"score": -1.0}  # tracks the best fold's val truth/preds

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    fold_model = clone(model)
    fold_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    va_pred = fold_model.predict(X.iloc[va_idx])

    score = f1_score(y.iloc[va_idx], va_pred, average="macro")
    fold_scores.append(score)

    if score > best["score"]:
        best = {
            "fold": fold,
            "score": score,
            "y_true": y.iloc[va_idx],
            "y_pred": va_pred,
        }

fold_scores = np.array(fold_scores)
print("Macro F1 per fold:", np.round(fold_scores, 4))
print(f"Mean macro F1: {fold_scores.mean():.4f} (+/- {fold_scores.std():.4f})")
print(f"Best fold: #{best['fold']} (macro F1 = {best['score']:.4f})")

Macro F1 per fold: [0.7474 0.7792 0.7669 0.7555 0.7545]
Mean macro F1: 0.7607 (+/- 0.0112)
Best fold: #1 (macro F1 = 0.7792)


In [6]:
# Classification report for the best candidate (highest macro F1 fold),
# evaluated on that fold's held-out validation split.
print(f"Classification report — best fold #{best['fold']} (macro F1 = {best['score']:.4f})\n")
print(classification_report(best["y_true"], best["y_pred"], digits=4))

Classification report — best fold #1 (macro F1 = 0.7792)

              precision    recall  f1-score   support

           0     0.7613    0.8087    0.7843       690
           1     0.7994    0.7504    0.7741       701

    accuracy                         0.7793      1391
   macro avg     0.7803    0.7795    0.7792      1391
weighted avg     0.7805    0.7793    0.7791      1391



In [7]:
# Fit on all training data, predict the test set
model.fit(X, y)
test_preds = model.predict(X_test)

In [8]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds.astype(int),
})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv", submission.shape)
print(submission[TARGET].value_counts().to_dict())
submission.head()

Wrote submission.csv (1739, 2)
{0: 951, 1: 788}


,GuestID,Churned
0,154038,1
1,620160,0
2,655103,0
3,126993,1
4,635228,0
